# 👑 KCS Sovereign-R1 — Modal Cloud GPU Deployment
**Architect**: Kandakatla Mahesh  
**Firm**: Kandakatla Capital Systems (KCS)  
**Platform**: Modal.com Serverless GPU (T4 / A10G)  

This notebook deploys the institutional `KCS Sovereign-R1` model to Modal.com in 1-click.

In [ ]:
# Step 1: Install Modal client in Colab
!pip install -q modal
print("[+] Modal CLI installed successfully!")

In [ ]:
# Step 2: Authenticate Modal
# Run this cell to authenticate your Modal account via browser link
!modal setup

In [ ]:
# Step 3: Write KCS Sovereign Modal Server App
modal_script = '''
import modal

kcs_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm>=0.6.0",
        "fastapi[standard]>=0.115.0",
        "huggingface_hub",
        "pydantic"
    )
)

app = modal.App("kcs-sovereign-r1", image=kcs_image)
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

SYSTEM_PROMPT = (
    "You are KCS Sovereign-R1, the state-of-the-art quantitative trading artificial intelligence "
    "engineered by Kandakatla Mahesh, Founder of Kandakatla Capital Systems (KCS). "
    "Evaluate market data across Equity, F&O, Commodities, Forex, and Crypto using structured "
    "mathematical reasoning (<think> ... </think>), strictly enforce risk preservation, and emit "
    "deterministic institutional execution setups."
)

@app.cls(
    gpu="T4",
    scaledown_window=300,
    timeout=600,
)
class KCSSovereignEngine:
    @modal.enter()
    def load_model(self):
        from vllm import LLM
        self.llm = LLM(
            model=MODEL_ID,
            tensor_parallel_size=1,
            max_model_len=4096,
            gpu_memory_utilization=0.90,
            trust_remote_code=True,
        )

    @modal.fastapi_endpoint(method="POST")
    def chat_completions(self, request: dict):
        from vllm import SamplingParams
        messages = request.get("messages", [])
        max_tokens = request.get("max_tokens", 512)
        temperature = request.get("temperature", 0.6)
        has_system = any(m.get("role") == "system" for m in messages)
        if not has_system:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        prompt = ""
        for m in messages:
            role = m.get("role", "user")
            content = m.get("content", "")
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"
        sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens, stop=["<|im_end|>", "<|endoftext|>"])
        outputs = self.llm.generate([prompt], sampling_params)
        output_text = outputs[0].outputs[0].text
        return {
            "choices": [{
                "message": {"role": "assistant", "content": output_text}
            }]
        }
'''
with open("kcs_modal_server.py", "w") as f:
    f.write(modal_script)
print("[+] Modal app script generated!")

In [ ]:
# Step 4: Deploy to Modal Cloud!
!modal deploy kcs_modal_server.py
print("\n[👑] KCS Sovereign-R1 is now LIVE on Modal Cloud!")